# Day 3: Errors and Causes
Input: `data/logs.parquet` from Day 1, the two nginx error logs, and `nginx_reload.log`.
Honeypot is excluded everywhere except section 5.

In [1]:
import re
from pathlib import Path
import pandas as pd
import plotly.express as px


In [2]:
df = pd.read_parquet('data/logs.parquet')
df['path'] = df.uri.str.split('?').str[0]
real = df[df.subdomain != 'a-honeypot']
errors = real[real.status_code >= 500]

def most_common(s):
    s=s.dropna()
    return s.mode().iloc[0] if len(s) else None

len(real), len(errors)

(859114, 19891)

## 1. 5xx per service: down or failing sometimes?

In [3]:
errors.status_code.value_counts()

status_code
502    18599
500     1040
504      250
503        2
Name: count, dtype: int64

**What each 5xx code means**
- 500: internal server error. The app answered, but with an error (a bug in the app).
- 502: bad gateway. nginx could not get a valid answer from the app, usually connection refused because the app is not running.
- 503: service unavailable. Overloaded or in maintenance.
- 504: gateway timeout. nginx waited too long for the app and gave up.

So 502 and 503 point to an app that is down, 504 to an app that is slow, 500 to broken code.
Here 502 dominates (18,599 of 19,891), so most errors are apps that are not running.

In [4]:
g = errors.groupby(['domain', 'subdomain'])

services = pd.DataFrame({
    'errors': g.size(),
    'main_code': g.status_code.agg(most_common),
    'hours_with_5xx': g.datetime.agg(lambda s: s.dt.floor('h').nunique()),
    'first': g.datetime.min(),
    'last': g.datetime.max(),
})

services['total'] = real.groupby(['domain', 'subdomain']).size()
services['share'] = (services.errors / services.total).round(3)
services['state'] = services.share.map(lambda s: 'down' if s >= 0.9 else 'failing sometimes')
services = services[services.errors >= 100].sort_values('errors', ascending=False)
services

errors  main_code  hours_with_5xx  \
domain       subdomain                                         
humblebee.ai speakology     11874        502             194   
             outline         2474        502               6   
             customs-ocr     1375        502              27   
             chatbot          784        502              10   
             jnj              713        500               6   
             smart-office     630        502              51   
hbai.dev     outline          487        500              28   
             secrets          323        502              14   
humblebee.ai repo-holmes      284        502              50   
hbai.dev     langfuse         225        502               8   
humblebee.ai hstaff           210        502               4   
hbai.dev     homepage         148        504               3   
             servers          104        502               5   

                                              first                      last  \
domain       subdomain                                                          
humblebee.ai speakology   2026-10-21 14:41:25+09:00 2026-10-29 15:12:51+09:00   
             outline      2026-10-27 18:26:03+09:00 2026-10-28 18:05:02+09:00   
             customs-ocr  2026-10-25 18:40:55+09:00 2026-10-29 15:12:48+09:00   
             chatbot      2026-10-27 18:26:32+09:00 2026-10-28 23:18:33+09:00   
             jnj          2026-10-26 04:10:02+09:00 2026-10-28 18:05:36+09:00   
             smart-office 2026-10-22 10:24:21+09:00 2026-10-28 18:38:35+09:00   
hbai.dev     outline      2026-10-21 16:46:28+09:00 2026-10-29 14:57:26+09:00   
             secrets      2026-10-21 16:46:48+09:00 2026-10-28 20:48:15+09:00   
humblebee.ai repo-holmes  2026-10-21 14:50:00+09:00 2026-10-29 12:55:25+09:00   
hbai.dev     langfuse     2026-10-21 16:46:34+09:00 2026-10-28 20:47:44+09:00   
humblebee.ai hstaff       2026-10-24 14:21:20+09:00 2026-10-28 17:44:54+09:00   
hbai.dev     homepage     2026-10-27 00:47:06+09:00 2026-10-27 10:38:27+09:00   
             servers      2026-10-27 00:36:55+09:00 2026-10-28 15:25:52+09:00   

                           total  share              state  
domain       subdomain                                      
humblebee.ai speakology    11888  0.999               down  
             outline       28235  0.088  failing sometimes  
             customs-ocr   25395  0.054  failing sometimes  
             chatbot       23138  0.034  failing sometimes  
             jnj           81922  0.009  failing sometimes  
             smart-office    630  1.000               down  
hbai.dev     outline       45412  0.011  failing sometimes  
             secrets       83674  0.004  failing sometimes  
humblebee.ai repo-holmes     285  0.996               down  
hbai.dev     langfuse      36899  0.006  failing sometimes  
humblebee.ai hstaff        61792  0.003  failing sometimes  
hbai.dev     homepage      66742  0.002  failing sometimes  
             servers       23087  0.005  failing sometimes

`errors` says **how many** errors a service had. `hours_with_5xx` says **how long** they lasted.
- speakology: 11,874 errors over 194 hours, about the whole log period. Down all week.
- outline: 2,474 errors packed into 6 hours. A short outage.
- repo-holmes: only 284 errors, but over 50 hours and a 99.6% share. Few people call it, yet it is down.

Counting errors alone would rank outline above repo-holmes, but repo-holmes is the one that is really dead.

In [5]:
top = services.index.get_level_values('subdomain')[:6]
hourly = (errors[errors.subdomain.isin(top)]
          .set_index('datetime')
          .groupby('subdomain')['request_id']
          .resample('1h').count()
          .reset_index(name='errors')
          )

px.line(hourly, x='datetime', y='errors', color='subdomain', title='5xx per hour, top 6 services')

**Reading the chart**
- speakology is a flat, constant line: down all the time.
- jnj spikes once, on Oct 26: a problem in that one app.
- On Oct 28 around 14:00 several services spike at the same moment. Separate apps do not break at the same second by chance, so something they share failed (host, proxy, or network). Section 2 checks this.

### Who is hitting the services that are down?

In [6]:
down = services[services.state == 'down'].index.get_level_values('subdomain')
(real[real.subdomain.isin(down)]
    .groupby(['subdomain', 'path'])
    .agg(requests=('request_id', 'size'), main_agent=('http_user_agent', most_common))
    .sort_values('requests', ascending=False)
    .head(8))

requests  \
subdomain    path                             
speakology   /                        11658   
repo-holmes  /api/github/webhooks       285   
speakology   /robots.txt                137   
smart-office /                           50   
             /robots.txt                 38   
speakology   /agent                      34   
             /favicon.ico                13   
smart-office /.git/config                 8   

                                                                          main_agent  
subdomain    path                                                                     
speakology   /                                                     Uptime-Kuma/2.0.2  
repo-holmes  /api/github/webhooks                            GitHub-Hookshot/6037803  
speakology   /robots.txt           Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...  
smart-office /                     Mozilla/5.0 (compatible; InternetMeasurement/1...  
             /robots.txt           Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...  
speakology   /agent                                       Python/3.10 aiohttp/3.13.2  
             /favicon.ico          Mozilla/5.0 (Linux; Android 10; K) AppleWebKit...  
smart-office /.git/config          Mozilla/5.0 AppleWebKit/537.36 (KHTML, like Ge...

## 2. Error bursts: many services failing at once
Group 5xx into 10-minute buckets. A burst that hits many services at the same moment points to something shared, not one broken app.

**Reading the bursts**
- Oct 26 04:10: 584 errors but only 3 services. Almost all are jnj 500s (559 in 20 minutes): a bug or crash inside jnj, since the app answered with an error.
- Oct 28 13:40 and 17:40: 11 and 12 services failing in the same 10 minutes, mostly 502. outline alone has 1,645 x 502 between 13:35 and 14:35. Something shared went down.
- Oct 27 18:20: 504 timeouts on hstaff, humblebee.ai, api-hstaff and access. Upstreams too slow, not down.

In [7]:
ts = errors.set_index('datetime').sort_index()
bursts = ts.resample('10min').agg({'request_id': 'size', 'subdomain': 'nunique'})
bursts.columns = ['errors', 'services']
bursts.sort_values('errors', ascending=False).head(10)


,errors,services
datetime,,
2026-10-26 04:10:00+09:00,584,3
2026-10-28 13:40:00+09:00,445,11
2026-10-28 14:20:00+09:00,393,5
2026-10-28 14:00:00+09:00,372,5
2026-10-28 14:10:00+09:00,367,5
2026-10-28 13:50:00+09:00,341,5
2026-10-28 17:50:00+09:00,322,5
2026-10-28 17:40:00+09:00,306,12
2026-10-28 14:30:00+09:00,300,5


In [8]:
def burst(start, minutes):
    start = pd.Timestamp(start, tz='Asia/Seoul')
    window = ts.loc[start:start + pd.Timedelta(minutes=minutes)]
    return window.groupby(['subdomain', 'status_code']).size().sort_values(ascending=False).head(6)

burst('2026-10-28 13:35', 60)

subdomain    status_code
outline      502            1645
chatbot      502             110
jnj          502             106
speakology   502              78
hstaff       502              68
customs-ocr  502              56
dtype: int64

In [9]:
burst('2026-10-26 04:05', 20)

subdomain     status_code
jnj           500            559
speakology    502             23
smart-office  502             14
dtype: int64

In [10]:
burst('2026-10-27 18:20', 40)

subdomain     status_code
hstaff        504            57
speakology    502            38
humblebee.ai  504            21
repo-holmes   502            10
api-hstaff    504             7
access        504             5
dtype: int64

In [11]:
burst('2026-10-27 13:20', 40)

subdomain   status_code
speakology  502            40
outline     500            12
dtype: int64

## 3. The error logs (hstaff, api-hstaff)

Format: `time [level] pid#tid: *conn message, client: X, server: X, request: "X", upstream: "X", host: "X"`.
`upstream` is optional, so its group is wrapped in `(?:...)?`. Each message is labelled by the first matching pattern in `ERROR_TYPES`.

In [12]:
ERR_RE = re.compile(
    r'(?P<time>\S+ \S+) \[(?P<level>\w+)\] \d+#\d+: (?:\*\d+ )?(?P<message>.*?), '
    r'client: (?P<client>[^,]+), server: (?P<server>[^,]+), '
    r'request: "(?P<request>[^"]*)"'
    r'(?:, upstream: "(?P<upstream>[^"]*)")?'
)

ERROR_TYPES = [
    ('refused', r'Connection refused'),
    ('timeout_connect', r'timed out.*while connecting'),
    ('timeout_read', r'timed out.*while reading'),
    ('premature_close', r'prematurely closed'),
    ('forbidden', r'forbidden')
]

def error_types(message):
    for label, pattern in ERROR_TYPES:
        if re.search(pattern, message):
            return label
    return 'other'

In [13]:
rows, bad = [], 0
for path in sorted(Path('nginx-logs').glob('*/*/*.error.log')):
    for line in open(path, encoding='utf-8', errors='replace'):
        m = ERR_RE.match(line)
        if m:
            rows.append({**m.groupdict(), 'subdomain': path.parts[2]})
        else:
            bad += 1

err = pd.DataFrame(rows)
err['time'] = pd.to_datetime(err.time, format='%Y/%m/%d %H:%M:%S').dt.tz_localize('Asia/Seoul')
err['type'] = err.message.map(error_types)
err['upstream_host'] = err.upstream.str.extract(r'//([^/]+)')[0]
len(err), bad

(352, 0)

In [14]:
err.groupby(['subdomain', 'type']).size().unstack(fill_value=0)

type,forbidden,premature_close,refused,timeout_connect,timeout_read
subdomain,,,,,
api-hstaff,0,0,12,7,0
hstaff,11,34,110,100,78


In [15]:
err.groupby(['upstream_host', 'type']).size().unstack(fill_value=0)

type,premature_close,refused,timeout_connect,timeout_read
upstream_host,,,,
100.100.1.30:21840,34,108,99,78
100.100.1.30:5173,0,2,1,0
100.100.1.30:8090,0,12,7,0


In [16]:
err.groupby([err.subdomain, err.time.dt.date]).size().unstack(fill_value=0)

time,2026-10-23,2026-10-24,2026-10-27,2026-10-28
subdomain,,,,
api-hstaff,0,0,7,12
hstaff,10,86,82,155


In [17]:
err.request.str.split().str[1].str.split('?').str[0].value_counts().head(5)

request
/mcp                             325
/                                  6
/api/v1/rise/assignments/my        3
/api/v1/reports/activity-logs      2
/api/v1/profile/                   2
Name: count, dtype: int64

**Error log findings**
- 352 errors parsed, 0 lines failed.
- Every error goes to one upstream host, 100.100.1.30 (a Tailscale machine). 319 of them to port 21840, which serves `/mcp` (325 requests).
- Types: 122 refused, 107 connect timeouts, 78 read timeouts, 34 premature close, 11 forbidden.
- Refused means the MCP service was not listening (restarting or down). Read timeouts on `/mcp` fit long MCP streams being cut by nginx `proxy_read_timeout`.

## 4. Do errors follow nginx reloads?
Compare 5xx in the 30 minutes before and after each reload.

In [18]:
reloads = pd.to_datetime(
    [line.split(': ', 1)[1].strip() for line in open('nginx-logs/nginx_reload.log') if line.strip()]
).tz_convert('Asia/Seoul')

window = pd.Timedelta('30min')
all_ts = real.set_index('datetime').sort_index()
pd.DataFrame([{
    'reload': r,
    'errors_before': (all_ts.loc[r - window:r].status_code >= 500).sum(),
    'errors_after': (all_ts.loc[r:r + window].status_code >= 500).sum(),
} for r in reloads])

,reload,errors_before,errors_after
0,2026-10-21 14:38:03+09:00,0,30
1,2026-10-22 10:02:35+09:00,29,34
2,2026-10-26 02:32:49+09:00,34,36


**Reload findings**
- Errors before and after each reload are about equal (29 vs 34, 34 vs 36).
- The first reload (Oct 21 14:38) is the start of the logs, so "before" has no data.
- Reloads are not a cause.

## 5. Scanners by behaviour
User agents hide scanners: most fake a browser. Behaviour does not lie: probing secret files and admin pages.

In [19]:
SUSPICIOUS = r'/\.env|/\.git|/wp-|/@fs/|/phpmyadmin|\.php$|/\.aws|/actuator|/server-status|/\.ds_store|/cgi-bin'
df['suspicious'] = df.path.str.contains(SUSPICIOUS, case=False, na=False)
df.groupby(df.subdomain == 'a-honeypot').suspicious.agg(['sum', 'mean']).rename(index={True: 'honeypot', False: 'real'})

,sum,mean
subdomain,,
real,5692,0.006625
honeypot,158666,0.482689


In [20]:
real = df[df.subdomain != 'a-honeypot']
by_ip = real.groupby('client_address').agg(
    requests=('request_id', 'size'),
    suspicious=('suspicious', 'sum'),
    share_404=('status_code', lambda s: round((s == 404).mean(), 3)),
    paths=('path', 'nunique'),
    subdomains=('subdomain', 'nunique'),
)
scanners = by_ip[by_ip.suspicious >= 10].sort_values('suspicious', ascending=False)
len(scanners), scanners.requests.sum()

(122, np.int64(18520))

In [21]:
# last part of each IP masked: real visitor IPs stay out of the shared notebook
scanners.head(10).rename(index=lambda ip: re.sub(r'[^.:]+$', 'x', ip))

,requests,suspicious,share_404,paths,subdomains
client_address,,,,,
20.194.17.x,533,528,0.503,157,3
185.177.72.x,2077,483,0.960,2058,1
54.206.82.x,395,385,0.577,140,2
103.59.160.x,231,228,0.494,77,1
45.94.31.x,180,162,0.000,19,1
3.27.159.x,140,136,0.814,140,1
15.134.134.x,140,136,0.814,140,1
74.225.146.x,134,133,0.963,132,1
185.177.72.x,585,120,0.858,186,1


In [22]:
real[real.suspicious].status_code.value_counts().head(6)

status_code
404    3866
308    1219
502     469
200     104
403      23
499       8
Name: count, dtype: int64

**Scanner findings**
- 122 IPs sent at least 10 suspicious requests each, 18,520 requests in total.
- The honeypot takes 158,666 suspicious requests (48% of its traffic), real services 5,692 (0.7%). So 96.5% of all probes hit the honeypot.
- Most probes on real services get 404 (3,866) or a redirect (1,219). 104 got 200: checked in section 6.

## 6. A 200 on a secret path: leak or not?
A 200 alone proves nothing. Many apps answer every unknown path with the same page (a catch-all).
Test: if many **different** secret paths return the **same** size, it is a catch-all page, not the file.

In [23]:
hits = real[real.suspicious & (real.status_code == 200)]
home = real[(real.path == '/') & (real.status_code == 200)].groupby('subdomain').response_body_size.median()

leak_check = hits.groupby('subdomain').agg(
    hits=('request_id', 'size'),
    different_paths=('path', 'nunique'),
    sizes=('response_body_size', 'nunique'),
    median_size=('response_body_size', 'median'),
)
leak_check['home_size'] = home
leak_check

,hits,different_paths,sizes,median_size,home_size
subdomain,,,,,
humblebee.ai,33,11,15,13343.0,28708.0
mail,5,5,1,1182.0,1182.0
status,66,17,1,608.0,0.0


In [24]:
hits[hits.subdomain == 'humblebee.ai'].groupby('path').response_body_size.agg(['count', 'min', 'max'])

,count,min,max
path,,,
/admin/.git/config,2,13352,13352
/admin/adminer.php,1,13338,13338
/admin/config.php,2,0,13343
/admin/controller/extension/extension/ultra.php,4,13383,13385
/admin/db.php,1,13337,13337
/admin/info.php,1,13339,13339
/admin/phpinfo.php,17,13340,13344
/admin/phpmyadmin.php,1,13344,13344
/admin/settings.php,1,13343,13343


## 7. Likely causes
| Group | Evidence | Likely cause |
|---|---|---|
| speakology down | 11,874 x 502 over 194 hours, the whole log period | App behind nginx is not running |
| smart-office, repo-holmes down | 502 share near 100%. repo-holmes traffic is GitHub webhooks | Upstream stopped. GitHub webhook deliveries are lost |
| Oct 28 13:35 to 14:35 and 17:35 to 18:05 | outline 1,645 x 502, up to 12 services failing in the same 10 minutes | Something shared went down (host, proxy, or network). Needs host logs to confirm |
| Oct 26 04:10 | jnj 559 x 500 in 20 minutes | Bug or crash inside jnj. Check jnj app logs at that time |
| Oct 27 18:20 | 504 on hstaff, humblebee.ai, api-hstaff, access | Upstreams too slow |
| hstaff error log | 352 errors, all to 100.100.1.30, 319 to port 21840 (`/mcp`) | MCP service restarting (refused) and long streams cut by nginx read timeout. Check `proxy_read_timeout` for `/mcp` |
| nginx reloads | errors before and after about equal | Not a cause |
| Scanners | 122 IPs, 18,520 requests, 96.5% of probes hit the honeypot | Normal internet noise |
| 200 on secret paths | `status`: 17 paths, 1 size. `mail`: same size as its homepage. `humblebee.ai` `/admin/*`: 11 paths, all about 13.3 KB | Catch-all pages, no evidence of a leak. Check `/admin/config.php` by hand (one 200 with 0 bytes) |